# Basic Language Model in Python

This notebook demonstrates how to create a basic language model using Python. The model processes a corpus of sentences and computes a language model, which is essentially a table showing the probability that a word follows a given prefix in the corpus.

## 1. Define the Corpus

In [1]:
corpus = [
    'the cat sat on the mat',
    'the cat sat on the chair',
    'the cat ate the fish',
    'the dog sat on the mat',
    'the dog ate the bone',
    'the dog chased the cat',
]

## 2. Tokenize the Corpus

In [2]:
# Create an empty list to store the tokenized sentences
tokenized_corpus = []

# Go through each sentence in the corpus
for sentence in corpus:
    # Split the sentence into a list of words and add it to our list
    tokenized_corpus.append(sentence.split())

## 3. Create Prefix-Word Table

In [3]:
# This dictionary will map each prefix to a dictionary of word counts
prefix_word_table = {}

# Go through each sentence in the tokenized corpus
for sentence in tokenized_corpus:
    # Go through each position in the sentence (except the last word)
    for i in range(len(sentence) - 1):
        # Build the prefix by joining all words from the start up to position i
        prefix = ' '.join(sentence[:i + 1])
        # The next word is the word right after the prefix
        word = sentence[i + 1]
        # If we haven't seen this prefix before, create an empty dictionary for it
        if prefix not in prefix_word_table:
            prefix_word_table[prefix] = {}
        # If we haven't seen this word after this prefix before, set its count to 0
        if word not in prefix_word_table[prefix]:
            prefix_word_table[prefix][word] = 0
        # Add 1 to the count of this word following this prefix
        prefix_word_table[prefix][word] += 1

## 4. Normalize Prefix-Word Table

In [4]:
# Go through each prefix and its word counts
for prefix, word_counts in prefix_word_table.items():
    # Calculate the total number of times any word followed this prefix
    total_count = sum(word_counts.values())
    # Divide each word's count by the total to get a probability
    for word, count in word_counts.items():
        prefix_word_table[prefix][word] = count / total_count

## 5. Visualize the Prefix-Word Table

In [5]:
#@title **Prefix-Word Probability Table** { display-mode: "form" }
import pandas as pd

all_words = sorted(set(w for probs in prefix_word_table.values() for w in probs))
data = []
for prefix, word_probs in prefix_word_table.items():
    row = {'Prefix': prefix}
    for w in all_words:
        row[w] = word_probs.get(w, 0.0)
    data.append(row)

df = pd.DataFrame(data).set_index('Prefix')

def color_cell(val):
    if val == 0:
        return 'color: #ddd; text-align: center;'
    return f'background-color: rgba(78,121,167,{val}); color: #222; font-weight: bold; text-align: center;'

df.style.format(lambda v: f'{v:.0%}' if v > 0 else '-').map(color_cell)

,ate,bone,cat,chair,chased,dog,fish,mat,on,sat,the
Prefix,,,,,,,,,,,
the,-,-,50%,-,-,50%,-,-,-,-,-
the cat,33%,-,-,-,-,-,-,-,-,67%,-
the cat sat,-,-,-,-,-,-,-,-,100%,-,-
the cat sat on,-,-,-,-,-,-,-,-,-,-,100%
the cat sat on the,-,-,-,50%,-,-,-,50%,-,-,-
the cat ate,-,-,-,-,-,-,-,-,-,-,100%
the cat ate the,-,-,-,-,-,-,100%,-,-,-,-
the dog,33%,-,-,-,33%,-,-,-,-,33%,-
the dog sat,-,-,-,-,-,-,-,-,100%,-,-


## 6. Generate Sentence from Prefix

In [9]:
# Import the random module (used to sample the next word based on probabilities)
import random

def generate_sentence(prefix):
    # Start the sentence with the given prefix, split into a list of words
    sentence = prefix.split()
    # Keep generating words until we can't continue
    while True:
        # Build the current prefix by joining all words generated so far
        current_prefix = ' '.join(sentence)
        # If the current prefix is not in the table, stop generating
        if current_prefix not in prefix_word_table:
            break
        # Get the probability distribution for words that can follow this prefix
        next_word_probs = prefix_word_table[current_prefix]
        # Extract the list of possible next words
        words = list(next_word_probs.keys())
        # Extract the corresponding probabilities
        probs = list(next_word_probs.values())
        # Pick a random number between 0 and 1
        r = random.random()
        # Walk through the probabilities to find which word was "selected"
        cumulative = 0
        for word, p in zip(words, probs):
            cumulative += p
            if r <= cumulative:
                next_word = word
                break
        # Add the chosen word to the sentence
        sentence.append(next_word)
    # Join all words into a single string and return the sentence
    return ' '.join(sentence)

# Generate a sentence starting with "the"
print(generate_sentence('the cat'))

the cat sat on the mat
